In [ ]:
import sys
sys.path.append("../")

#
# TO BE USED WITH the Godot project in ../ackermann
# Missione completa:
# 1) il carrello parte dal punto A
# 2) va al punto B, dove c'e' lo scaffale
# 3) il braccio prende il pacco
# 4) il braccio va in posizione di non ingombro
# 5) il carrello torna al punto A
# 6) il braccio si abbassa e scarica il pacco
#

from lib.system.cart import *
from lib.dds.dds import *
from lib.utils.time import *
from lib.system.controllers import *
from lib.data.dataplot import *
from lib.system.polar import *
from manipulator_control import *
from lib.system.trajectory import *

# -------------------------
# DDS / Godot middleware
# -------------------------
dds = DDS()
dds.start()
dds.subscribe(['tick', 'TerrainHeight', 'TerrainSlope'])

# -------------------------
# Robot: carrello + braccio
# -------------------------
arm = FourJointsManipulatorControl()

terrain = MeasuredTerrainProfile()
cart2d = AckermannSlopeLoad(
    cart_mass=10,
    friction=0.8,
    wheel_radius=0.5,
    wheelbase=2.0,
    terrain=terrain,
)

linear_speed_controller = PID_Controller(50.0, 10.0, 0, 100.0)
polar_position = Polar2DController(
    1.0, 3.0,                 # kp posizione, velocita' max
    5.0, math.radians(40)     # kp angolo, sterzo max
)

virtual_robot = StraightLine2DMotion(1.0, 1.5, 1.5)

# -------------------------
# Missione
# -------------------------
# A = scarico
# B = scaffale / carico
point_a = (0.0, 0.0)
point_b = (0.0, -25.0)

zone_radius = 0.35
arrival_threshold = zone_radius
packages_to_load = [5.0, 5.0, 5.0]

cart2d.set_pose(point_a[0], point_a[1])

# IMPORTANTE: adesso si carica al punto B e si scarica al punto A.
load_zone = PackageTransferZone('B_shelf_load', point_b, zone_radius, 'load', packages_to_load)
unload_zone = PackageTransferZone('A_unload', point_a, zone_radius, 'unload')
cargo_events = []

# -------------------------
# Pose del braccio
# -------------------------
# Sono pose locali rispetto al carrello, non coordinate mondo.
# Modifica questi numeri in base alla posizione reale dello scaffale in Godot.
ARM_SAFE_POSE = (0.20, 0.00, 0.55, math.radians(-90))       # posizione chiusa/non ingombrante
ARM_PICK_APPROACH = (0.25, 0.45, 0.40, math.radians(-90))   # davanti/sopra il pacco sullo scaffale
ARM_PICK_DOWN = (0.25, 0.45, 0.12, math.radians(-90))       # scende per prendere il pacco
ARM_DROP_APPROACH = (0.25, 0.00, 0.35, math.radians(-90))   # sopra la zona di scarico
ARM_DROP_DOWN = (0.25, 0.00, 0.08, math.radians(-90))       # scende per scaricare

ARM_POS_TOL = 0.04
ARM_ANGLE_TOL = math.radians(7)


def arm_is_near(arm, target_pose, pos_tol=ARM_POS_TOL, angle_tol=ARM_ANGLE_TOL):
    x_ref, y_ref, z_ref, a_ref = target_pose
    x, y, z, a = arm.get_pose()

    pos_error = math.sqrt(
        (x - x_ref) ** 2 +
        (y - y_ref) ** 2 +
        (z - z_ref) ** 2
    )

    angle_error = abs(a - a_ref)
    return pos_error < pos_tol and angle_error < angle_tol


def set_arm_target_or_warn(target_pose, label):
    ok = arm.set_target(*target_pose)
    if not ok:
        print('WARNING: target braccio non raggiungibile:', label, target_pose)
    return ok


# All'inizio il braccio va in posizione sicura.
set_arm_target_or_warn(ARM_SAFE_POSE, 'ARM_SAFE_POSE')

# Macchina a stati della missione
DRIVE_TO_B = 'DRIVE_TO_B'
PICK_APPROACH = 'PICK_APPROACH'
PICK_DOWN = 'PICK_DOWN'
PICK_LIFT_SAFE = 'PICK_LIFT_SAFE'
DRIVE_TO_A = 'DRIVE_TO_A'
DROP_APPROACH = 'DROP_APPROACH'
DROP_DOWN = 'DROP_DOWN'
DROP_LIFT_SAFE = 'DROP_LIFT_SAFE'
DONE = 'DONE'

state = DRIVE_TO_B
state_changed = True
virtual_robot.start_motion(point_a, point_b)

# -------------------------
# Plot
# -------------------------
vdp = DataPlotter()
vdp.set_x("time (seconds)")
vdp.add_y("target_speed", "Target Linear Speed")
vdp.add_y("current_speed", "Current Linear Speed")

sdp = DataPlotter()
sdp.set_x("time (seconds)")
sdp.add_y("angle", "Steering Angle")

cdp = DataPlotter()
cdp.set_x("time (seconds)")
cdp.add_y("payload", "Payload Mass (kg)")
cdp.add_y("height", "Height (m)")
cdp.add_y("slope", "Slope (degrees)")
cdp.add_y("mission_state", "Mission State")

state_id = {
    DRIVE_TO_B: 0,
    PICK_APPROACH: 1,
    PICK_DOWN: 2,
    PICK_LIFT_SAFE: 3,
    DRIVE_TO_A: 4,
    DROP_APPROACH: 5,
    DROP_DOWN: 6,
    DROP_LIFT_SAFE: 7,
    DONE: 8,
}

# -------------------------
# Main loop
# -------------------------
t = Time()
t.start()

while t.get() < 80:

    dds.wait('tick')
    delta_t = t.elapsed()

    terrain_height = dds.read('TerrainHeight')
    terrain_slope = dds.read('TerrainSlope')
    if terrain_height is not None and terrain_slope is not None:
        terrain.update(cart2d.s, terrain_height, terrain_slope)

    pose = cart2d.get_pose()
    pose_3d = cart2d.get_pose_3d()
    v, w = cart2d.get_speed()

    target_speed = 0.0
    steering_angle = 0.0
    torque = 0.0

    # -------------------------
    # 1) Vai allo scaffale B
    # -------------------------
    if state == DRIVE_TO_B:
        target_x, target_y = virtual_robot.evaluate(delta_t)
        target_speed, steering_angle = polar_position.evaluate(delta_t, target_x, target_y, pose)
        torque = linear_speed_controller.evaluate(delta_t, target_speed - v)

        dist_b = math.hypot(pose[0] - point_b[0], pose[1] - point_b[1])
        if dist_b <= arrival_threshold:
            print('Arrivato al punto B: inizio presa dallo scaffale')
            state = PICK_APPROACH
            set_arm_target_or_warn(ARM_PICK_APPROACH, 'ARM_PICK_APPROACH')
            torque = 0.0
            steering_angle = 0.0

    # -------------------------
    # 2) Porta il braccio davanti/sopra il pacco
    # -------------------------
    elif state == PICK_APPROACH:
        torque = 0.0
        steering_angle = 0.0

        if arm_is_near(arm, ARM_PICK_APPROACH):
            print('Braccio sopra il pacco: scendo per prendere')
            state = PICK_DOWN
            set_arm_target_or_warn(ARM_PICK_DOWN, 'ARM_PICK_DOWN')

    # -------------------------
    # 3) Scendi, prendi il pacco, poi torna sicuro
    # -------------------------
    elif state == PICK_DOWN:
        torque = 0.0
        steering_angle = 0.0

        if arm_is_near(arm, ARM_PICK_DOWN):
            cargo_event = load_zone.process(cart2d)
            if cargo_event is not None:
                cargo_events.append(cargo_event)
                print('cargo_event:', cargo_event)
            else:
                print('Nessun cargo_event in load: controlla posizione del carrello o zone_radius')

            print('Pacco preso: porto il braccio in posizione di non ingombro')
            state = PICK_LIFT_SAFE
            set_arm_target_or_warn(ARM_SAFE_POSE, 'ARM_SAFE_POSE_AFTER_PICK')

    # -------------------------
    # 4) Quando il braccio e' sicuro, parti verso A
    # -------------------------
    elif state == PICK_LIFT_SAFE:
        torque = 0.0
        steering_angle = 0.0

        if arm_is_near(arm, ARM_SAFE_POSE):
            print('Braccio sicuro: parto verso il punto A per scaricare')
            virtual_robot.start_motion(point_b, point_a)
            state = DRIVE_TO_A

    # -------------------------
    # 5) Vai al punto A
    # -------------------------
    elif state == DRIVE_TO_A:
        target_x, target_y = virtual_robot.evaluate(delta_t)
        target_speed, steering_angle = polar_position.evaluate(delta_t, target_x, target_y, pose)
        torque = linear_speed_controller.evaluate(delta_t, target_speed - v)

        dist_a = math.hypot(pose[0] - point_a[0], pose[1] - point_a[1])
        if dist_a <= arrival_threshold:
            print('Arrivato al punto A: preparo lo scarico')
            state = DROP_APPROACH
            set_arm_target_or_warn(ARM_DROP_APPROACH, 'ARM_DROP_APPROACH')
            torque = 0.0
            steering_angle = 0.0

    # -------------------------
    # 6) Braccio sopra la zona di scarico
    # -------------------------
    elif state == DROP_APPROACH:
        torque = 0.0
        steering_angle = 0.0

        if arm_is_near(arm, ARM_DROP_APPROACH):
            print('Braccio sopra la zona di scarico: scendo')
            state = DROP_DOWN
            set_arm_target_or_warn(ARM_DROP_DOWN, 'ARM_DROP_DOWN')

    # -------------------------
    # 7) Scendi e scarica
    # -------------------------
    elif state == DROP_DOWN:
        torque = 0.0
        steering_angle = 0.0

        if arm_is_near(arm, ARM_DROP_DOWN):
            cargo_event = unload_zone.process(cart2d)
            if cargo_event is not None:
                cargo_events.append(cargo_event)
                print('cargo_event:', cargo_event)
            else:
                print('Nessun cargo_event in unload: controlla posizione del carrello o zone_radius')

            print('Pacco scaricato: riporto il braccio in posizione sicura')
            state = DROP_LIFT_SAFE
            set_arm_target_or_warn(ARM_SAFE_POSE, 'ARM_SAFE_POSE_AFTER_DROP')

    # -------------------------
    # 8) Fine
    # -------------------------
    elif state == DROP_LIFT_SAFE:
        torque = 0.0
        steering_angle = 0.0

        if arm_is_near(arm, ARM_SAFE_POSE):
            print('Missione completata')
            state = DONE

    elif state == DONE:
        torque = 0.0
        steering_angle = 0.0

    # Aggiorna dinamica carrello e braccio.
    # Il carrello riceve torque=0 e steering=0 durante le fasi del braccio.
    cart2d.evaluate(delta_t, torque, steering_angle)
    arm.evaluate(delta_t)

    # Rileggi pose dopo evaluate.
    pose_3d = cart2d.get_pose_3d()
    v, w = cart2d.get_speed()
    theta0, theta1, theta2, theta3 = arm.get_joint_angles()
    arm_x, arm_y, arm_z, arm_a = arm.get_pose()

    # CargoPhase per Godot/debug:
    # 0 = guida verso B; 1 = presa; 2 = guida verso A; 3 = scarico; 4 = fine
    if state in [PICK_APPROACH, PICK_DOWN, PICK_LIFT_SAFE]:
        cargo_phase = 1.0
    elif state == DRIVE_TO_A:
        cargo_phase = 2.0
    elif state in [DROP_APPROACH, DROP_DOWN, DROP_LIFT_SAFE]:
        cargo_phase = 3.0
    elif state == DONE:
        cargo_phase = 4.0
    else:
        cargo_phase = 0.0

    # -------------------------
    # Publish DDS verso Godot
    # -------------------------
    dds.publish('X', pose_3d[0], DDS.DDS_TYPE_FLOAT)
    dds.publish('Y', pose_3d[1], DDS.DDS_TYPE_FLOAT)
    dds.publish('Z', pose_3d[2], DDS.DDS_TYPE_FLOAT)
    dds.publish('Theta', pose_3d[3], DDS.DDS_TYPE_FLOAT)
    dds.publish('Slope', pose_3d[4], DDS.DDS_TYPE_FLOAT)
    dds.publish('PayloadMass', cart2d.get_payload_mass(), DDS.DDS_TYPE_FLOAT)
    dds.publish('CargoPhase', cargo_phase, DDS.DDS_TYPE_FLOAT)
    dds.publish('MissionState', float(state_id[state]), DDS.DDS_TYPE_FLOAT)

    # Topic del braccio. Se in Godot hai gia' nomi diversi, mappa questi topic nel tuo script Godot.
    dds.publish('theta0', theta0, DDS.DDS_TYPE_FLOAT)
    dds.publish('theta1', theta1, DDS.DDS_TYPE_FLOAT)
    dds.publish('theta2', theta2, DDS.DDS_TYPE_FLOAT)
    dds.publish('theta3', theta3, DDS.DDS_TYPE_FLOAT)
    dds.publish('arm_x', arm_x, DDS.DDS_TYPE_FLOAT)
    dds.publish('arm_y', arm_y, DDS.DDS_TYPE_FLOAT)
    dds.publish('arm_z', arm_z, DDS.DDS_TYPE_FLOAT)
    dds.publish('arm_a', arm_a, DDS.DDS_TYPE_FLOAT)

    # Se il tuo vecchio progetto Godot ascolta x/y/z/a per il manipolatore,
    # tengo anche questi alias per compatibilita'.
    dds.publish('x', arm_x, DDS.DDS_TYPE_FLOAT)
    dds.publish('y', arm_y, DDS.DDS_TYPE_FLOAT)
    dds.publish('z', arm_z, DDS.DDS_TYPE_FLOAT)
    dds.publish('a', arm_a, DDS.DDS_TYPE_FLOAT)

    # -------------------------
    # Plot data
    # -------------------------
    vdp.append_x(t.get())
    vdp.append_y("current_speed", v)
    vdp.append_y("target_speed", target_speed)

    sdp.append_x(t.get())
    sdp.append_y("angle", math.degrees(steering_angle))

    cdp.append_x(t.get())
    cdp.append_y("payload", cart2d.get_payload_mass())
    cdp.append_y("height", pose_3d[2])
    cdp.append_y("slope", math.degrees(pose_3d[4]))
    cdp.append_y("mission_state", state_id[state])

    if state == DONE:
        break

# -------------------------
# Final report
# -------------------------
vdp.plot()
sdp.plot()
cdp.plot()

final_pose = cart2d.get_pose_3d()
final_error_a = math.hypot(final_pose[0] - point_a[0], final_pose[1] - point_a[1])
reached_a = final_error_a <= arrival_threshold
mission_complete = state == DONE and reached_a and cart2d.get_payload_mass() == 0

print('A unload:', point_a, 'B shelf/load:', point_b, 'zone_radius_m:', zone_radius)
print('cargo_events:', cargo_events)
print('final_pose:', final_pose, 'final_error_from_A_m:', final_error_a)
print('payload_kg:', cart2d.get_payload_mass(), 'reached_A:', reached_a, 'mission_complete:', mission_complete)

dds.stop()
